# Testing file 
### where we evaluate Zhang's models using the test set

## Preliminaries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.data import Dataset


from util.load_data import load_data
from util.evaluation import *
from models.zhang.models import FairLogisticRegression
from models.zhang.learning import train_loop as zhang_train

/Users/lffpl/Projects/falsb/env/falsb/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
batch_size = 64
epochs = 100
lr = 0.001

In [3]:
cv_seeds = [13, 29, 42, 55, 73]

## Load data

In [4]:
data_name = 'heart'

In [5]:
x, y, a = load_data(data_name)
raw_data = (x, y, a)

In [6]:
xdim = x.shape[1]
ydim = y.shape[1]
adim = a.shape[1]
zdim = 8

In [7]:
print(xdim, ydim, adim, zdim)

25 1 1 8


In [8]:
print(len(x))

1025


## Result file

In [9]:
header = "model_name", "cv_seed", "clas_acc", "dp", "deqodds", "deqopp", "trade_dp", "trade_deqodds", "trade_deqopp", "TN_a0", "FP_a0", "FN_a0", "TP_a0", "TN_a1", "FP_a1", "FN_a1", "TP_a1"
results = []

## Testing loop
#### Each model is evalueted 5 times
#### In the end of each iteration we save the result

### Zhang for DP

In [10]:
fairdef = 'DemPar'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)

    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4DP', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc


2026-01-03 18:07:00.985286: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 1 | 0.7792488932609558 | 1.0375992059707642 | 0.5142045454545454 | 0.3125


2026-01-03 18:07:01.234789: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 2 | 0.7614266872406006 | 1.0277727842330933 | 0.5142045454545454 | 0.3125
> 3 | 0.7449370622634888 | 1.0210235118865967 | 0.5142045454545454 | 0.3125


...

> 64 | 0.35063090920448303 | 0.7922500371932983 | 0.8607954545454546 | 0.35511363636363635
> 65 | 0.34946301579475403 | 0.7896106243133545 | 0.8607954545454546 | 0.3565340909090909
> 66 | 0.3483448922634125 | 0.7869442701339722 | 0.8607954545454546 | 0.35795454545454547
> 67 | 0.34729480743408203 | 0.7843230366706848 | 0.8607954545454546 | 0.359375
> 68 | 0.34625130891799927 | 0.7816863656044006 | 0.8622159090909091 | 0.35795454545454547
> 69 | 0.34521910548210144 | 0.7790341377258301 | 0.8622159090909091 | 0.359375
> 70 | 0.34423622488975525 | 0.7763901948928833 | 0.8622159090909091 | 0.3622159090909091
> 71 | 0.34326449036598206 | 0.7737297415733337 | 0.8636363636363636 | 0.359375
> 72 | 0.34232237935066223 | 0.7711055874824524 | 0.8636363636363636 | 0.35511363636363635
> 73 | 0.34145069122314453 | 0.7685065269470215 | 0.8636363636363636 | 0.35511363636363635
> 74 | 0.340597540140152 | 0.7659351229667664 | 0.8636363636363636 | 0.35795454545454547
> 75 | 0.3397749662399292 | 0.7634075

2026-01-03 18:07:32.498386: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 27 | 0.4479013681411743 | 0.8620110750198364 | 0.8565340909090909 | 0.3125
> 28 | 0.43933814764022827 | 0.8596341013908386 | 0.8607954545454546 | 0.3125
> 29 | 0.4311331808567047 | 0.857262134552002 | 0.8551136363636364 | 0.3125
> 30 | 0.4235859811306 | 0.8548537492752075 | 0.8579545454545454 | 0.3125
> 31 | 0.41628748178482056 | 0.8524686098098755 | 0.8636363636363636 | 0.3125
> 32 | 0.40937504172325134 | 0.8500974178314209 | 0.8622159090909091 | 0.3125
> 33 | 0.402691513299942 | 0.8477463722229004 | 0.8607954545454546 | 0.3125
> 34 | 0.3962445855140686 | 0.8454140424728394 | 0.859375 | 0.3125
> 35 | 0.3901820480823517 | 0.843033492565155 | 0.8607954545454546 | 0.3125
> 36 | 0.38442733883857727 | 0.8407323360443115 | 0.8607954545454546 | 0.3125
> 37 | 0.3790323734283447 | 0.8383767604827881 | 0.8607954545454546 | 0.3125
> 38 | 0.37387174367904663 | 0.8360273838043213 | 0.8622159090909091 | 0.3125
> 39 | 0.3688674569129944 | 0.8337016701698303 | 0.8622159090909091 | 0.3125
> 40 | 0.3

2026-01-03 18:08:04.093659: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 54 | 0.4622936248779297 | 0.851807713508606 | 0.875 | 0.32386363636363635
> 55 | 0.46119919419288635 | 0.8494811058044434 | 0.8735795454545454 | 0.3210227272727273
> 56 | 0.4601624011993408 | 0.847185492515564 | 0.8735795454545454 | 0.328125
> 57 | 0.4591817855834961 | 0.84492027759552 | 0.875 | 0.3338068181818182
> 58 | 0.45827358961105347 | 0.8429083824157715 | 0.875 | 0.3380681818181818
> 59 | 0.4575064182281494 | 0.8409035801887512 | 0.875 | 0.34375
> 60 | 0.4568079710006714 | 0.8389101624488831 | 0.875 | 0.35511363636363635
> 61 | 0.45615535974502563 | 0.8369395136833191 | 0.875 | 0.3565340909090909
> 62 | 0.4555455148220062 | 0.8349837064743042 | 0.875 | 0.36363636363636365
> 63 | 0.45505037903785706 | 0.8329769372940063 | 0.8735795454545454 | 0.3622159090909091
> 64 | 0.4546072781085968 | 0.8310117125511169 | 0.8721590909090909 | 0.36363636363636365
> 65 | 0.45420682430267334 | 0.8292043805122375 | 0.8721590909090909 | 0.37073863636363635
> 66 | 0.45384109020233154 | 0.8273606

### Zhang for Eq Odds

In [11]:
fairdef = 'EqOdds'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)
    
    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc
> 1 | 0.7792488932609558 | 1.0331954956054688 | 0.5142045454545454 | 0.3125
> 2 | 0.7614268064498901 | 1.0191833972930908 | 0.5142045454545454 | 0.3125
> 3 | 0.7449373006820679 | 1.008413553237915 | 0.5142045454545454 | 0.3125
> 4 | 0.7289673686027527 | 0.9979944229125977 | 0.5142045454545454 | 0.3125
> 5 | 0.713445782661438 | 0.9878983497619629 | 0.5142045454545454 | 0.3125
> 6 | 0.6983110308647156 | 0.9781819581985474 | 0.5142045454545454 | 0.3125


2026-01-03 18:09:09.701911: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 7 | 0.68362957239151 | 0.9688712954521179 | 0.5142045454545454 | 0.3125
> 8 | 0.6696087718009949 | 0.9599617123603821 | 0.5142045454545454 | 0.3125
> 9 | 0.6560931205749512 | 0.9514222741127014 | 0.5142045454545454 | 0.3125
> 10 | 0.6430057287216187 | 0.9433035850524902 | 0.5142045454545454 | 0.3125
> 11 | 0.6303329467773438 | 0.9355694651603699 | 0.5170454545454546 | 0.3125
> 12 | 0.618117094039917 | 0.9282621741294861 | 0.5355113636363636 | 0.3125
> 13 | 0.6062530875205994 | 0.9213862419128418 | 0.5795454545454546 | 0.3125
> 14 | 0.5947708487510681 | 0.9148600101470947 | 0.6136363636363636 | 0.3125
> 15 | 0.5836734771728516 | 0.9087498188018799 | 0.6477272727272727 | 0.3125
> 16 | 0.5729749798774719 | 0.9030302166938782 | 0.6903409090909091 | 0.3125
> 17 | 0.5626572370529175 | 0.8976448774337769 | 0.7258522727272727 | 0.3125
> 18 | 0.5526508688926697 | 0.8926388621330261 | 0.7585227272727273 | 0.3125
> 19 | 0.5430084466934204 | 0.8879843950271606 | 0.7869318181818182 | 0.3125
> 20 

### Zhang for Eq Opp

In [12]:
fairdef = 'EqOpp'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)
    
    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4EqOpp', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc
> 1 | 0.7801633477210999 | 0.46891048550605774 | 0.5142045454545454 | 0.3125
> 2 | 0.7631853818893433 | 0.46393269300460815 | 0.5142045454545454 | 0.3125
> 3 | 0.7474570274353027 | 0.46044206619262695 | 0.5142045454545454 | 0.3125
> 4 | 0.7321443557739258 | 0.45702946186065674 | 0.5142045454545454 | 0.3125
> 5 | 0.7174326777458191 | 0.45369166135787964 | 0.5142045454545454 | 0.3125
> 6 | 0.7029968500137329 | 0.4504033327102661 | 0.5142045454545454 | 0.3125
> 7 | 0.688927173614502 | 0.44718965888023376 | 0.5142045454545454 | 0.3125
> 8 | 0.675365686416626 | 0.4440488815307617 | 0.5142045454545454 | 0.3125
> 9 | 0.6623038053512573 | 0.44098126888275146 | 0.5142045454545454 | 0.3125
> 10 | 0.6497164964675903 | 0.43797963857650757 | 0.5142045454545454 | 0.3125
> 11 | 0.6374963521957397 | 0.43502819538116455 | 0.515625 | 0.3125
> 12 | 0.625650942325592 | 0.43214091658592224 | 0.5241477272727273 | 0.3125
> 13 | 0.6141492128372192 | 0.4293

2026-01-03 18:11:21.311658: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 14 | 0.6030198931694031 | 0.4265442192554474 | 0.5852272727272727 | 0.3125
> 15 | 0.5922322273254395 | 0.4238000512123108 | 0.6221590909090909 | 0.3125
> 16 | 0.5817896127700806 | 0.4211268424987793 | 0.6519886363636364 | 0.3125
> 17 | 0.5717025995254517 | 0.4185226261615753 | 0.7017045454545454 | 0.3125
> 18 | 0.561954915523529 | 0.41599276661872864 | 0.7258522727272727 | 0.3125
> 19 | 0.5525420904159546 | 0.4135058522224426 | 0.7514204545454546 | 0.3125
> 20 | 0.5433993935585022 | 0.4111054539680481 | 0.7784090909090909 | 0.3125
> 21 | 0.5345772504806519 | 0.40878140926361084 | 0.8011363636363636 | 0.3125
> 22 | 0.5260896682739258 | 0.40650224685668945 | 0.8153409090909091 | 0.3125
> 23 | 0.5179356932640076 | 0.40429726243019104 | 0.8210227272727273 | 0.3125
> 24 | 0.5100747346878052 | 0.40207040309906006 | 0.8352272727272727 | 0.3125
> 25 | 0.5025370121002197 | 0.39989912509918213 | 0.8338068181818182 | 0.3125
> 26 | 0.495267391204834 | 0.3977917432785034 | 0.8394886363636364 | 0.

## Saving into DF then CSV

In [13]:
result_df = pd.DataFrame(results, columns=header)
result_df

,model_name,cv_seed,clas_acc,dp,deqodds,deqopp,trade_dp,trade_deqodds,trade_deqopp,TN_a0,FP_a0,FN_a0,TP_a0,TN_a1,FP_a1,FN_a1,TP_a1
0,Zhang4DP,13,0.863281,0.687132,0.866313,0.803333,0.765200,0.864795,0.832229,20.0,2.0,1.0,59.0,83.0,16.0,16.0,59.0
1,Zhang4DP,29,0.882812,0.688094,0.928205,0.862013,0.773385,0.904940,0.872289,18.0,3.0,1.0,55.0,88.0,14.0,12.0,65.0
2,Zhang4DP,42,0.824219,0.711605,0.909110,0.850673,0.763783,0.864586,0.837237,19.0,6.0,3.0,53.0,84.0,22.0,14.0,55.0
3,Zhang4DP,55,0.855469,0.714213,0.904669,0.870877,0.778485,0.879381,0.863104,17.0,3.0,1.0,56.0,82.0,22.0,11.0,64.0
4,Zhang4DP,73,0.859375,0.707435,0.931377,0.891963,0.776038,0.893928,0.875366,20.0,4.0,3.0,66.0,78.0,19.0,10.0,56.0
5,Zhang4EqOdds,13,0.863281,0.687132,0.866313,0.803333,0.765200,0.864795,0.832229,20.0,2.0,1.0,59.0,83.0,16.0,16.0,59.0
6,Zhang4EqOdds,29,0.882812,0.688094,0.928205,0.862013,0.773385,0.904940,0.872289,18.0,3.0,1.0,55.0,88.0,14.0,12.0,65.0
7,Zhang4EqOdds,42,0.824219,0.711605,0.909110,0.850673,0.763783,0.864586,0.837237,19.0,6.0,3.0,53.0,84.0,22.0,14.0,55.0
8,Zhang4EqOdds,55,0.851562,0.719800,0.899862,0.870877,0.780157,0.875046,0.861112,17.0,3.0,1.0,56.0,81.0,23.0,11.0,64.0
9,Zhang4EqOdds,73,0.859375,0.707435,0.931377,0.891963,0.776038,0.893928,0.875366,20.0,4.0,3.0,66.0,78.0,19.0,10.0,56.0


In [14]:
result_df.to_csv(f'{data_name}-result/zhang-{epochs}.csv')